# Day 03 — LLM Judges: The Model Behind the Score

**Module 1 · Foundations**

In Day 02, we created an `LLMTestCase`.

But a test case by itself does not tell us whether the AI response is good or bad.

We need something to **evaluate the response**.

In this notebook, we will:

1. Understand what an LLM judge is.
2. Create a judge model with DeepEval.
3. Understand the role of the judge.
4. Understand why the judge configuration matters.

> **Core idea:** An LLM judge is an LLM used to evaluate another AI system's output.

## 1. What Is an LLM Judge?

Consider this interaction:

```text
User Question
      ↓
   AI System
      ↓
Generated Answer

## 2. Create Our Judge

For this course, we will use a Groq-hosted model as the judge.

The judge will be accessed through an OpenAI-compatible API endpoint.

In [1]:
import os

from dotenv import load_dotenv
from deepeval.models import LocalModel

load_dotenv()

assert os.getenv("GROQ_API_KEY"), "GROQ_API_KEY not found."

judge = LocalModel(
    model="openai/gpt-oss-120b",
    base_url="https://api.groq.com/openai/v1",
    api_key=os.environ["GROQ_API_KEY"],
    temperature=0,
)

print("Judge:", judge.get_model_name())

Judge: openai/gpt-oss-120b (Local Model)


## 3. What Did We Just Create?

We created a model whose job in our evaluation system is to act as the **judge**.

The important configuration is:

| Setting | Purpose |
|---|---|
| `model` | The LLM used for evaluation |
| `base_url` | The API endpoint |
| `api_key` | Authentication |
| `temperature=0` | More consistent judging |

The application model and the judge have different responsibilities:

```text
Application Model
    ↓
Generates the answer

Judge Model
    ↓
Evaluates the answer

## 4. Why Use a Separate Judge?

Imagine the same model generating and grading its own answer:

```text
Same Model
   ├── Generate
   └── Judge

## 5. Why Temperature = 0?

Our evaluation system should produce reasonably consistent judgments.

If the judge itself behaves highly randomly, the same test case could receive different scores simply because the judge changed its response.

Therefore, we use:

```python
temperature=0

## 6. How DeepEval Uses the Judge

We will later create metrics such as:

- Answer Relevancy
- Faithfulness
- Correctness
- Summarization

Conceptually, the process looks like this:

```text
LLMTestCase
     ↓
   Metric
     ↓
 Judge Prompt
     ↓
 LLM Judge
     ↓
 Score + Reason
     ↓
 Threshold
     ↓
 PASS / FAIL

## 7. A Simple Mental Model

Think of the system as having two LLM roles:

```text
┌──────────────────────┐
│   Application LLM    │
│                      │
│ "Generate an answer" │
└──────────┬───────────┘
           │
           ↓
      Actual Output
           │
           ↓
┌──────────────────────┐
│      Judge LLM       │
│                      │
│ "Evaluate the answer"│
└──────────┬───────────┘
           │
           ↓
       Score + Reason